# KITTI-360 4-camera 3DGUT + Fast3DGS Colab

Pipeline thực hành:

- 4 camera KITTI-360: 2 pinhole + 2 fisheye.
- LiDAR Velodyne thật để khởi tạo Gaussian.
- Pose chính thức `cam0_to_world.txt`.
- Rasterization `gsplat` với `with_ut=True` cho fisheye.
- `SelectiveAdam` theo kiểu Fast3DGS.
- Full dataset và output lưu trên Google Drive.

Lần sau chỉ cần chạy lần lượt các cell. Không chạy lại cell tải dataset nếu archive đã tồn tại.


In [ ]:
#@title 2. Mount Drive + cài dependency
from google.colab import drive
drive.mount('/content/drive')

%pip install -q ninja jaxtyping rich scipy opencv-python-headless
%pip install -q --no-build-isolation "gsplat==1.5.3"


In [ ]:
#@title 2. Kiểm tra / tải KITTI-360 nếu thiếu
from pathlib import Path

loader_source = 'import os\nimport sys\nimport urllib.request\nfrom pathlib import Path\n\nBASE = "https://huggingface.co/datasets/danny042/KITTI-360-fisheye/resolve/main"\nROOT = Path("/content/drive/MyDrive/datasets/kitti360_full")\nROOT.mkdir(parents=True, exist_ok=True)\n\nFILES = [\n    "calibration/calib_cam_to_pose.txt",\n    "calibration/calib_cam_to_velo.txt",\n    "calibration/calib_sick_to_velo.txt",\n    "calibration/image_02.yaml",\n    "calibration/image_03.yaml",\n    "calibration/perspective.txt",\n    "ImageSets-dapetr-v49-port/kitti360_dapetr_infos_train.pkl",\n    "ImageSets-dapetr-v49-port/kitti360_dapetr_infos_val.pkl",\n    "ImageSets-rosa-6class/kitti360_rosa_infos_train.pkl",\n    "ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl",\n]\nfor seq in ("0000", "0002", "0003", "0004", "0005", "0006", "0007", "0009", "0010"):\n    name = f"2013_05_28_drive_{seq}_sync"\n    FILES.extend(\n        [f"data_2d_raw/{name}_image_{cam}.tar" for cam in ("00", "01", "02", "03")]\n    )\n    FILES.append(f"data_3d_raw/{name}.tar")\n\n\ndef download(rel):\n    dst = ROOT / rel\n    if dst.exists() and dst.stat().st_size > 0:\n        print(f"[SKIP] {rel} already exists", flush=True)\n        return True\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE}/{rel}"\n    part = dst.with_suffix(dst.suffix + ".part")\n    start = part.stat().st_size if part.exists() else 0\n    req = urllib.request.Request(url, headers={"User-Agent": "kitti360-full-loader"})\n    if start:\n        req.add_header("Range", f"bytes={start}-")\n    try:\n        with urllib.request.urlopen(req, timeout=120) as src:\n            mode = "ab" if start else "wb"\n            total = int(src.headers.get("Content-Length", "0")) + start\n            done = start\n            with part.open(mode) as out:\n                while True:\n                    block = src.read(8 * 1024 * 1024)\n                    if not block:\n                        break\n                    out.write(block)\n                    done += len(block)\n                    if total:\n                        print(f"[{done * 100 // total:3d}%] {rel}", flush=True)\n        part.replace(dst)\n        print(f"[DONE] {rel}", flush=True)\n    except Exception as exc:\n        print(f"[FAIL] {rel}: {exc}", flush=True)\n        return False\n    return True\n\n\nprint(f"Target: {ROOT}", flush=True)\nprint(f"Files: {len(FILES)}", flush=True)\nfailed = [rel for rel in FILES if not download(rel)]\nif failed:\n    print("FAILED FILES:", *failed, sep="\\n", flush=True)\n    sys.exit(1)\nprint("FULL KITTI360 ARCHIVE READY", flush=True)\n'
Path('/content/download_kitti360_full.py').write_text(loader_source, encoding='utf-8')
%run /content/download_kitti360_full.py

# Official KITTI-360 poses are a separate small archive.
pose_dst = Path('/content/drive/MyDrive/datasets/kitti360_full/data_poses.zip')
pose_url = 'https://s3.eu-central-1.amazonaws.com/avg-projects/KITTI-360/89a6bae3c8a6f789e12de4807fc1e8fdcf182cf4/data_poses.zip'
if pose_dst.exists() and pose_dst.stat().st_size > 1_000_000:
    print('[SKIP] data_poses.zip already exists')
else:
    import urllib.request
    print('[DOWNLOAD] data_poses.zip')
    urllib.request.urlretrieve(pose_url, pose_dst)
    print('[DONE] data_poses.zip', pose_dst.stat().st_size, 'bytes')


In [ ]:
#@title 3. Cấu hình
import os

# Archive trên Google Drive (chỉ cần sequence 0003 + calibration + pkl + data_poses.zip).
os.environ['KITTI360_ARCHIVE'] = '/content/drive/MyDrive/datasets/kitti360_full'
os.environ['KITTI360_N_FRAMES'] = '100'     # 10 để smoke test
os.environ['KITTI360_STEPS'] = '10000'      # PSNR gần như bão hòa sau ~5000-10000
os.environ['KITTI360_CAP_MAX'] = '1000000'  # T4 đủ bộ nhớ
os.environ['KITTI360_MASK'] = '1'           # bỏ qua thân xe/ngoài vòng tròn fisheye
os.environ['KITTI360_SSIM'] = '1'           # chậm hơn ~4x, +0.5-0.9 dB pinhole
os.environ['KITTI360_SKY'] = '1'            # quan trọng nhất: +4-5 dB
os.environ['KITTI360_RUN_NAME'] = 'run_notebook'
os.environ['KITTI360_SYNC_DIR'] = '/content/drive/MyDrive/datasets/kitti360_fast3dgut_runs'
os.environ['KITTI360_OUT'] = '/content/kitti360_cli_out'
print(os.environ['KITTI360_ARCHIVE'])


In [ ]:
#@title 4. Kiểm tra dataset và pose chính thức
from pathlib import Path
from zipfile import ZipFile

archive = Path(os.environ['KITTI360_ARCHIVE'])
required = [
    archive / 'ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl',
    archive / 'data_poses.zip',
]
for p in required:
    print(('OK  ' if p.exists() else 'MISS'), p)
if not all(p.exists() for p in required):
    raise FileNotFoundError('Thiếu archive hoặc data_poses.zip trên Drive.')
with ZipFile(required[1]) as z:
    name = '2013_05_28_drive_0003_sync/cam0_to_world.txt'
    print('Pose entries:', name in z.namelist())


In [ ]:
#@title 5. Ghi trainer 3DGUT + Fast3DGS-style (MCMC, mask, SSIM, sky)
from pathlib import Path
trainer_source = '"""KITTI-360 4-camera 3DGUT + Fast3DGS-style trainer (gsplat).\n\n- 2 pinhole (image_00/01) + 2 fisheye (image_02/03), real Velodyne LiDAR, official poses.\n- Gaussians initialised from LiDAR, colours sampled from the camera that saw the scan.\n- gsplat rasterization with 3DGUT (with_ut + with_eval3d) so fisheye is rendered exactly.\n- Fast3DGS-style SelectiveAdam: only Gaussians visible in the current view are updated.\n- gsplat MCMCStrategy: relocation of dead Gaussians and growth to a cap (no 2D gradients needed).\n\nEnv: KITTI360_N_FRAMES (100), KITTI360_STEPS (20000), KITTI360_MAX_GAUSSIANS (120000), KITTI360_CAP_MAX (1000000), KITTI360_SYNC_DIR, KITTI360_RUN_NAME, KITTI360_RESUME,\n     KITTI360_TEST_EVERY (10), KITTI360_SEQ, KITTI360_START, KITTI360_MAX_GAP, KITTI360_ARCHIVE, KITTI360_OUT (/content/kitti360_cli_out).\n"""\nimport json\nimport os\nimport pickle\nimport random\nimport shutil\nimport tarfile\nimport time\nimport urllib.request\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport cv2\nimport numpy as np\nimport torch\nfrom scipy.signal import medfilt\nfrom scipy.spatial import cKDTree\nfrom gsplat import rasterization\nfrom gsplat.optimizers import SelectiveAdam\nfrom gsplat.strategy import MCMCStrategy\n\nBASE = "https://huggingface.co/datasets/danny042/KITTI-360-fisheye/resolve/main"\nPOSES_URL = ("https://s3.eu-central-1.amazonaws.com/avg-projects/KITTI-360/"\n             "89a6bae3c8a6f789e12de4807fc1e8fdcf182cf4/data_poses.zip")\nSEQ = os.environ.get("KITTI360_SEQ", "2013_05_28_drive_0003_sync")\nSTART = int(os.environ.get("KITTI360_START", "0"))   # first frame id to consider\nMAX_GAP = int(os.environ.get("KITTI360_MAX_GAP", "5"))  # frames further apart than this break a run (huge = never)\nARCHIVE = Path(os.environ.get("KITTI360_ARCHIVE", "/content/kitti360_min"))\nOUT = Path(os.environ.get("KITTI360_OUT", "/content/kitti360_cli_out"))\nROOT = OUT / "work"   # scratch space inside the output directory: always writable, not Colab-specific\nN_FRAMES = int(os.environ.get("KITTI360_N_FRAMES", "100"))\nSTEPS = int(os.environ.get("KITTI360_STEPS", "20000"))\nMAX_GAUSSIANS = int(os.environ.get("KITTI360_MAX_GAUSSIANS", "120000"))\nCAP_MAX = int(os.environ.get("KITTI360_CAP_MAX", "1000000"))\nSEED = int(os.environ.get("KITTI360_SEED", "42"))\nRUN_NAME = os.environ.get("KITTI360_RUN_NAME", time.strftime("run_%Y%m%d_%H%M%S"))\nSYNC_DIR = os.environ.get("KITTI360_SYNC_DIR", "")      # e.g. a Drive folder; empty disables syncing\nRESUME = os.environ.get("KITTI360_RESUME", "0") == "1"   # continue RUN_NAME from its last checkpoint\nCKPT_EVERY = int(os.environ.get("KITTI360_CKPT_EVERY", "2500"))\nUSE_MASK = os.environ.get("KITTI360_MASK", "1") == "1"    # ignore the car hood / outside the fisheye circle\nUSE_SSIM = os.environ.get("KITTI360_SSIM", "0") == "1"    # loss = 0.8 L1 + 0.2 (1 - SSIM)\nSSIM_IMPL = os.environ.get("KITTI360_SSIM_IMPL", "auto")  # auto | fused | torch (fused = fused-ssim CUDA kernel)\nUSE_SKY = os.environ.get("KITTI360_SKY", "0") == "1"      # seed large Gaussians on a far sky dome\nMASK_ONLY = os.environ.get("KITTI360_MASK_ONLY", "0") == "1"\nPERSIST = float(os.environ.get("KITTI360_PERSIST", "0.5"))   # edge persistence above which a lower-image edge is static\nSKY_RADIUS, SKY_SCALE, SKY_PER_FRAME = 200.0, 12.0, 400\nMAX_SCALE = 40.0 if USE_SKY else 0.5\nLOG_PATH = os.environ.get("KITTI360_LOG", "/content/train.log")\nTEST_EVERY = int(os.environ.get("KITTI360_TEST_EVERY", "10"))\nSCALE = 2\nCAMERAS = [\n    ("CAM_FRONT", "image_00", "pinhole"),\n    ("CAM_FRONT_RIGHT", "image_01", "pinhole"),\n    ("CAM_BACK_RIGHT", "image_02", "fisheye"),\n    ("CAM_BACK_LEFT", "image_03", "fisheye"),\n]\nDEV = "cuda"\n\n\n# ------------------------------------------------------------------ data access\ndef fetch(url):\n    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})\n    return urllib.request.urlopen(req, timeout=120)\n\n\ndef ensure_file(rel, url=None, min_size=1):\n    dst = ARCHIVE / rel\n    if dst.exists() and dst.stat().st_size >= min_size:\n        return dst\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    with fetch(url or f"{BASE}/{rel}") as src, dst.open("wb") as out:\n        while True:\n            block = src.read(1 << 20)\n            if not block:\n                break\n            out.write(block)\n    return dst\n\n\ndef stream_extract(rel, suffix, wanted, attempts=3):\n    """Read wanted members from a local tar or stream it from Hugging Face (with retries)."""\n    local = ARCHIVE / rel\n    for attempt in range(attempts):\n        out = {}\n        try:\n            source = local.open("rb") if local.exists() else fetch(f"{BASE}/{rel}")\n            mode = "r:*" if local.exists() else "r|*"\n            with source, tarfile.open(fileobj=source, mode=mode) as tar:\n                for member in tar:\n                    if not member.isfile() or not member.name.endswith(suffix):\n                        continue\n                    stem = Path(member.name).stem\n                    if stem in wanted:\n                        out[stem] = tar.extractfile(member).read()\n                        if len(out) == len(wanted):\n                            break\n            return out\n        except Exception as exc:  # network hiccups while streaming multi-GB tars\n            print(f"[RETRY] {rel} attempt {attempt + 1}: {exc}", flush=True)\n            time.sleep(5)\n    raise RuntimeError(f"could not read {rel}")\n\n\ndef read_official_poses():\n    path = ensure_file("data_poses.zip", POSES_URL, min_size=1_000_000)\n    with ZipFile(path) as archive, archive.open(f"{SEQ}/cam0_to_world.txt") as stream:\n        poses = {}\n        for line in stream:\n            values = np.array(line.split(), dtype=np.float64)\n            if len(values) == 17:\n                poses[int(values[0])] = values[1:].reshape(4, 4)\n    return poses\n\n\n# ------------------------------------------------------------------ camera models\ndef mei_to_fisheye(xi, k1, k2):\n    """Fit gsplat\'s OpenCV-fisheye radial polynomial to the exact Mei model (<0.002 px error)."""\n    theta = np.linspace(0.02, 1.45, 256)\n    u = np.sin(theta) / (np.cos(theta) + xi)\n    u2 = u * u\n    r = (1 + xi) * u * (1 + k1 * u2 + k2 * u2 * u2)\n    X = np.column_stack([theta**2, theta**4, theta**6, theta**8])\n    k, *_ = np.linalg.lstsq(X, r / theta - 1, rcond=None)\n    return k.astype(np.float32)\n\n\ndef project_for_color(view, pts_world):\n    """Pixel coordinates of world points in a view (same camera model the renderer uses)."""\n    pc = pts_world @ view["T"][:3, :3].T + view["T"][:3, 3]\n    K = view["K_np"]\n    if view["model"] == "pinhole":\n        ok = pc[:, 2] > 0.5\n        z = np.where(ok, pc[:, 2], 1.0)\n        uv = np.column_stack([K[0, 0] * pc[:, 0] / z + K[0, 2], K[1, 1] * pc[:, 1] / z + K[1, 2]])\n        return uv, ok\n    norm_xy = np.linalg.norm(pc[:, :2], axis=1)\n    theta = np.arctan2(norm_xy, pc[:, 2])\n    ok = (np.linalg.norm(pc, axis=1) > 0.5) & (theta < 1.40)  # renderer limit is pi/2\n    t2 = theta * theta\n    r = view["radial_np"]\n    theta_d = theta * (1 + r[0] * t2 + r[1] * t2**2 + r[2] * t2**3 + r[3] * t2**4)\n    scale = np.where(norm_xy > 1e-9, theta_d / np.maximum(norm_xy, 1e-9), 1.0)\n    uv = np.column_stack([K[0, 0] * pc[:, 0] * scale + K[0, 2], K[1, 1] * pc[:, 1] * scale + K[1, 2]])\n    return uv, ok\n\n\ndef sample_colors(views_of_frame, pts_world):\n    """Colour from the first camera of the same frame that sees the point."""\n    colors = np.zeros((len(pts_world), 3), np.float32)\n    seen = np.zeros(len(pts_world), bool)\n    for view in views_of_frame:\n        uv, ok = project_for_color(view, pts_world)\n        h, w = view["image_np"].shape[:2]\n        inside = ok & ~seen & (uv[:, 0] >= 1) & (uv[:, 0] < w - 2) & (uv[:, 1] >= 1) & (uv[:, 1] < h - 2)\n        if not inside.any():\n            continue\n        px = np.round(uv[inside]).astype(int)\n        colors[inside] = view["image_np"][px[:, 1], px[:, 0]].astype(np.float32) / 255.0\n        seen |= inside\n    return colors, seen\n\n\n# ------------------------------------------------------------------ masks, SSIM\ndef static_masks(views):\n    """1 = supervise, 0 = ignore. The car hood and the black area outside the fisheye circle are\n    fixed in the camera frame. The hood\'s colours change with the light, but its *edges* stay at the\n    same pixels in every frame while scene edges move, so edge persistence finds it."""\n    masks = {}\n    for cam in ("image_02", "image_03"):\n        gray = np.stack([cv2.cvtColor(v["image_np"], cv2.COLOR_RGB2GRAY) for v in views if v["cam"] == cam]\n                        ).astype(np.float32)\n        sob = lambda g: np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))\n        mean_grad = np.mean([sob(g) for g in gray], axis=0)          # average edge strength per pixel\n        persistence = sob(gray.mean(0)) / (mean_grad + 1.0)          # ~1 where edges never move\n        h, w = mean_grad.shape\n        rows = np.arange(h)[:, None]\n        edges = ((persistence > PERSIST) & (mean_grad > 15) & (rows > 0.63 * h)).astype(np.uint8)\n        closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((31, 31), np.uint8))\n        # Hood = everything below the lowest-reaching persistent edge in each column.\n        top = np.array([np.flatnonzero(closed[:, x]).min() if closed[:, x].any() else h for x in range(w)], np.float32)\n        top = medfilt(top, 61)                       # single-column noise must not cut into the scene\n        hood = (np.arange(h)[:, None] >= top[None, :]).astype(np.uint8)\n        outside = ((gray.mean(0) < 12) & (gray.std(0) < 6)).astype(np.uint8)\n        bad = cv2.dilate(np.maximum(hood, outside), np.ones((9, 9), np.uint8))\n        masks[cam] = (1 - bad).astype(np.float32)\n        cv2.imwrite(str(OUT / f"mask_{cam}.png"), masks[cam] * 255)\n        cv2.imwrite(str(OUT / f"persist_{cam}.png"), np.clip(persistence * 255, 0, 255).astype(np.uint8))\n        sync(OUT / f"mask_{cam}.png")\n        print(f"[MASK] {cam}: supervising {masks[cam].mean() * 100:.0f}% of pixels", flush=True)\n    return masks\n\n\ndef make_ssim():\n    """Pick the SSIM kernel: the fused-ssim CUDA package when installed (accelerator), else conv2d."""\n    if SSIM_IMPL in ("auto", "fused"):\n        try:\n            from fused_ssim import fused_ssim\n            print("[ACCEL] SSIM: fused-ssim", flush=True)\n            return lambda a, b: fused_ssim(a.permute(2, 0, 1)[None], b.permute(2, 0, 1)[None])\n        except ImportError:\n            if SSIM_IMPL == "fused":\n                raise\n    print("[ACCEL] SSIM: torch conv2d", flush=True)\n    return ssim_fn\n\n\ndef ssim_fn(a, b, win=11, sigma=1.5):\n    x, y = a.permute(2, 0, 1)[None], b.permute(2, 0, 1)[None]\n    g = torch.exp(-((torch.arange(win, device=x.device).float() - win // 2) ** 2) / (2 * sigma**2))\n    g = g / g.sum()\n    k = (g[:, None] * g[None, :])[None, None].repeat(3, 1, 1, 1)\n    f = lambda t: torch.nn.functional.conv2d(t, k, padding=win // 2, groups=3)\n    mx, my = f(x), f(y)\n    sxx, syy, sxy = f(x * x) - mx**2, f(y * y) - my**2, f(x * y) - mx * my\n    c1, c2 = 0.01**2, 0.03**2\n    return (((2 * mx * my + c1) * (2 * sxy + c2)) / ((mx**2 + my**2 + c1) * (sxx + syy + c2))).mean()\n\n\n# ------------------------------------------------------------------ durable output\ndef sync(path):\n    """Copy a finished local file into SYNC_DIR/RUN_NAME atomically (tmp file, size check, rename)."""\n    if not SYNC_DIR:\n        return\n    path = Path(path)\n    dst_dir = Path(SYNC_DIR) / RUN_NAME\n    dst, tmp = dst_dir / path.name, dst_dir / (path.name + ".tmp")\n    for attempt in range(3):\n        try:\n            dst_dir.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(path, tmp)\n            if tmp.stat().st_size != path.stat().st_size:\n                raise IOError("size mismatch after copy")\n            os.replace(tmp, dst)\n            return\n        except Exception as exc:  # Drive FUSE can hiccup; training must not die because of it\n            print(f"[SYNC-RETRY] {path.name}: {exc}", flush=True)\n            time.sleep(5)\n    print(f"[SYNC-FAIL] {path.name} (local copy kept in {path.parent})", flush=True)\n\n\n# ------------------------------------------------------------------ export\ndef save_ply(path, means, quats, scales, opacity, colors):\n    names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2", "opacity",\n             "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"]\n    data = np.zeros(len(means), dtype=[(n, "<f4") for n in names])\n    values = np.column_stack([means, np.zeros_like(means), (colors - 0.5) / 0.28209479,\n                              opacity, scales, quats])\n    for i, name in enumerate(names):\n        data[name] = values[:, i]\n    with path.open("wb") as stream:\n        header = f"ply\\nformat binary_little_endian 1.0\\nelement vertex {len(data)}\\n"\n        header += "".join(f"property float {n}\\n" for n in names)\n        stream.write((header + "end_header\\n").encode("ascii"))\n        stream.write(data.tobytes())\n\n\ndef select_items(all_items, official):\n    """First run of N_FRAMES consecutive frames (gaps <= MAX_GAP) of SEQ from START that have poses."""\n    pool = sorted((x for x in all_items if SEQ in x["lidar_path"] and x["frame_id"] in official\n                   and x["frame_id"] >= START), key=lambda x: x["frame_id"])\n    runs, cur = [], []\n    for x in pool:\n        if cur and x["frame_id"] - cur[-1]["frame_id"] > MAX_GAP:\n            runs.append(cur)\n            cur = []\n        cur.append(x)\n    if cur:\n        runs.append(cur)\n    if not runs:\n        raise SystemExit(f"no frames with poses for {SEQ} from frame {START}")\n    run = next((r for r in runs if len(r) >= N_FRAMES), max(runs, key=len))\n    if len(run) < N_FRAMES:\n        print(f"[WARN] longest run in {SEQ} from frame {START} has {len(run)} frames (< {N_FRAMES})", flush=True)\n    return run[:N_FRAMES]\n\n\n# ------------------------------------------------------------------ main\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    ROOT.mkdir(parents=True, exist_ok=True)\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    import gsplat\n    config = dict(sequence=SEQ, start=START, mask=USE_MASK, ssim=USE_SSIM, sky=USE_SKY, run=RUN_NAME, seed=SEED, frames=N_FRAMES, steps=STEPS, cap_max=CAP_MAX,\n                  init_gaussians=MAX_GAUSSIANS, test_every=TEST_EVERY, image_scale=SCALE,\n                  torch=torch.__version__, gsplat=gsplat.__version__, resume=RESUME,\n                  gpu=torch.cuda.get_device_name(0))\n    (OUT / "config.json").write_text(json.dumps(config, indent=2))\n    sync(OUT / "config.json")\n    all_items = []\n    for split in ("val", "train"):   # val holds frames 12-282 of this sequence, train the rest\n        meta_path = ensure_file(f"ImageSets-rosa-6class/kitti360_rosa_infos_{split}.pkl")\n        with meta_path.open("rb") as stream:\n            all_items += pickle.load(stream)["data_list"]\n    official = read_official_poses()\n    items = select_items(all_items, official)\n    frame_ids = [x["frame_id"] for x in items]\n    config["frame_ids"] = frame_ids      # the renderer reads this, so video and model always use the same frames\n    (OUT / "config.json").write_text(json.dumps(config, indent=2))\n    sync(OUT / "config.json")\n    item_by_frame = {x["frame_id"]: x for x in items}\n    wanted = {f"{x:010d}" for x in frame_ids}\n    print(f"[DATA] {len(frame_ids)} frames: {frame_ids[0]}..{frame_ids[-1]}", flush=True)\n\n    t0 = time.perf_counter()\n    jobs = {cam: (f"data_2d_raw/{SEQ}_{cam}.tar", ".png") for _, cam, _ in CAMERAS}\n    jobs["lidar"] = (f"data_3d_raw/{SEQ}.tar", ".bin")\n    with ThreadPoolExecutor(len(jobs)) as pool:\n        futures = {k: pool.submit(stream_extract, rel, suf, wanted) for k, (rel, suf) in jobs.items()}\n        raw = {k: f.result() for k, f in futures.items()}\n    print(f"[DATA] downloaded in {time.perf_counter() - t0:.0f}s:",\n          {k: len(v) for k, v in raw.items()}, flush=True)\n\n    # World frame = official world, re-centred on the first frame for float32 precision.\n    front_lidar2cam = np.array(item_by_frame[frame_ids[0]]["images"]["CAM_FRONT"]["lidar2cam"], dtype=np.float64)\n    origin = official[frame_ids[0]][:3, 3].copy()\n    world_from_lidar = {}\n    for fid in frame_ids:\n        m = official[fid] @ front_lidar2cam     # cam0_to_world @ cam0_from_lidar (checked by projection_check)\n        m[:3, 3] -= origin\n        world_from_lidar[fid] = m\n\n    views = []\n    for fid in frame_ids:\n        item = item_by_frame[fid]\n        for key, cam, model in CAMERAS:\n            info = item["images"][key]\n            bgr = cv2.imdecode(np.frombuffer(raw[cam][f"{fid:010d}"], np.uint8), cv2.IMREAD_COLOR)\n            h, w = bgr.shape[0] // SCALE, bgr.shape[1] // SCALE\n            rgb = cv2.cvtColor(cv2.resize(bgr, (w, h), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)\n            k = np.array(info["camera_intrinsic"], dtype=np.float64)\n            radial = np.zeros(4, np.float32)\n            if model == "fisheye":\n                k1, k2, _, _, xi = info["distortion_coefficients"]\n                k[0, 0] /= 1 + xi\n                k[1, 1] /= 1 + xi\n                radial = mei_to_fisheye(xi, k1, k2)\n            k[0] /= SCALE\n            k[1] /= SCALE\n            lidar2cam = np.array(info["lidar2cam"], dtype=np.float64)\n            T = lidar2cam @ np.linalg.inv(world_from_lidar[fid])\n            views.append(dict(fid=fid, cam=cam, model=model, h=h, w=w, image_np=rgb, K_np=k,\n                              radial_np=radial, T=T))\n\n    masks = static_masks(views)     # always computed: evaluation uses it even when training ignores it\n    if MASK_ONLY:\n        print("[DONE] mask preview only", flush=True)\n        return\n\n    # ---- Gaussian initialisation: LiDAR geometry + colour from the camera that saw it.\n    rng = np.random.default_rng(42)\n    per_frame = max(500, int(MAX_GAUSSIANS * 1.5 / len(frame_ids)))\n    views_by_frame = {}\n    for v in views:\n        views_by_frame.setdefault(v["fid"], []).append(v)\n    pts_all, col_all = [], []\n    for fid in frame_ids:\n        scan = np.frombuffer(raw["lidar"][f"{fid:010d}"], np.float32).reshape(-1, 4)[:, :3].astype(np.float64)\n        dist = np.linalg.norm(scan, axis=1)\n        scan = scan[(dist > 2.5) & (dist < 80)]\n        world = scan @ world_from_lidar[fid][:3, :3].T + world_from_lidar[fid][:3, 3]\n        colors, seen = sample_colors(views_by_frame[fid], world)\n        world, colors = world[seen], colors[seen]          # keep only points some camera observed\n        pick = rng.choice(len(world), min(per_frame, len(world)), replace=False)\n        pts_all.append(world[pick])\n        col_all.append(colors[pick])\n    del raw\n    points = np.concatenate(pts_all).astype(np.float32)\n    colors0 = np.concatenate(col_all).astype(np.float32)\n    if len(points) > MAX_GAUSSIANS:\n        pick = rng.choice(len(points), MAX_GAUSSIANS, replace=False)\n        points, colors0 = points[pick], colors0[pick]\n    n = len(points)\n    knn = cKDTree(points).query(points, k=4)[0][:, 1:].mean(1).clip(0.03, 1.0).astype(np.float32)\n    if USE_SKY:\n        sky_pts, sky_col = [], []\n        for fid in frame_ids:\n            d = rng.normal(size=(SKY_PER_FRAME, 3))\n            d = d / np.linalg.norm(d, axis=1, keepdims=True)\n            d[:, 2] = np.abs(d[:, 2])\n            d = d[d[:, 2] > np.sin(np.radians(12))]        # world z is up: keep the upper sky\n            world = world_from_lidar[fid][:3, 3] + SKY_RADIUS * d\n            colors, seen = sample_colors(views_by_frame[fid], world)\n            sky_pts.append(world[seen])\n            sky_col.append(colors[seen])\n        sky_pts, sky_col = np.concatenate(sky_pts).astype(np.float32), np.concatenate(sky_col).astype(np.float32)\n        points = np.concatenate([points, sky_pts])\n        colors0 = np.concatenate([colors0, sky_col])\n        knn = np.concatenate([knn, np.full(len(sky_pts), SKY_SCALE, np.float32)])\n        n = len(points)\n        print(f"[INIT] +{len(sky_pts)} sky Gaussians on a {SKY_RADIUS:.0f} m dome", flush=True)\n    print(f"[INIT] {n} Gaussians from LiDAR, extent {np.ptp(points, axis=0).round(1)} m", flush=True)\n\n    centers = np.array([-v["T"][:3, :3].T @ v["T"][:3, 3] for v in views if v["cam"] == "image_00"])\n    # gsplat scales the position lr and the split/prune thresholds with scene_scale. A driving\n    # sequence is hundreds of metres long but its Gaussians are centimetres to decimetres, so\n    # the trajectory length (~370 m for 100 frames) must not be used directly.\n    trajectory = float(np.linalg.norm(centers - centers.mean(0), axis=1).max())\n    scene_scale = float(np.clip(1.1 * trajectory, 5.0, 20.0))\n    print(f"[INIT] scene_scale {scene_scale:.1f} m (trajectory radius {trajectory:.0f} m)", flush=True)\n\n    def logit(x):\n        return np.log(x / (1 - x))\n\n    params = torch.nn.ParameterDict({\n        "means": torch.nn.Parameter(torch.tensor(points, device=DEV)),\n        "quats": torch.nn.Parameter(torch.tensor(np.tile([1.0, 0, 0, 0], (n, 1)), dtype=torch.float32, device=DEV)),\n        "scales": torch.nn.Parameter(torch.tensor(np.log(knn[:, None].repeat(3, 1)), device=DEV)),\n        "opacities": torch.nn.Parameter(torch.full((n,), float(logit(0.3)), device=DEV)),\n        "colors": torch.nn.Parameter(torch.tensor(logit(colors0.clip(0.02, 0.98)), device=DEV)),\n    })\n    lrs = {"means": 1.6e-4 * scene_scale, "quats": 1e-3, "scales": 5e-3, "opacities": 5e-2, "colors": 2.5e-2}\n    optimizers = {k: SelectiveAdam([{"params": [params[k]], "lr": lr, "name": k}], eps=1e-15, betas=(0.9, 0.999))\n                  for k, lr in lrs.items()}\n    start_step = 0\n    ckpt_dir = (Path(SYNC_DIR) / RUN_NAME) if SYNC_DIR else OUT\n    if RESUME and (ckpt_dir / "ckpt.pt").exists():\n        ck = torch.load(ckpt_dir / "ckpt.pt", map_location=DEV)\n        params = torch.nn.ParameterDict({k: torch.nn.Parameter(v.to(DEV)) for k, v in ck["params"].items()})\n        optimizers = {k: SelectiveAdam([{"params": [params[k]], "lr": lr, "name": k}], eps=1e-15,\n                                       betas=(0.9, 0.999)) for k, lr in lrs.items()}\n        for k, opt in optimizers.items():\n            opt.load_state_dict(ck["opt"][k])\n        start_step = int(ck["step"])\n        print(f"[RESUME] step {start_step}, {len(params[\'means\'])} Gaussians from {ckpt_dir / \'ckpt.pt\'}", flush=True)\n        del ck\n    # 3DGUT (with_eval3d) never touches means2d, so gradient-based densification (DefaultStrategy)\n    # has nothing to read. MCMC relocates dead Gaussians and grows towards cap_max instead.\n    strategy = MCMCStrategy(cap_max=CAP_MAX, refine_start_iter=500, refine_stop_iter=int(STEPS * 0.8),\n                            refine_every=100, min_opacity=0.005, verbose=False)\n    strategy.check_sanity(params, optimizers)\n    state = strategy.initialize_state()\n\n    # ---- Move images to the GPU as uint8 (400 views of 100 frames fit on a T4).\n    for v in views:\n        v["image"] = torch.from_numpy(v["image_np"]).to(DEV)\n        v["view"] = torch.tensor(v["T"], dtype=torch.float32, device=DEV)[None]\n        v["K"] = torch.tensor(v["K_np"], dtype=torch.float32, device=DEV)[None]\n        v["radial"] = torch.tensor(v["radial_np"], dtype=torch.float32, device=DEV)[None]\n        m = masks.get(v["cam"], np.ones((v["h"], v["w"]), np.float32))\n        v["eval_mask"] = torch.from_numpy(m).to(DEV)[..., None]\n        v["mask"] = v["eval_mask"] if USE_MASK else torch.ones_like(v["eval_mask"])   # loss mask\n    test_fids = set(frame_ids[TEST_EVERY - 1::TEST_EVERY]) if TEST_EVERY else set()\n    train = [v for v in views if v["fid"] not in test_fids]\n    test = [v for v in views if v["fid"] in test_fids]\n    print(f"[SPLIT] train views {len(train)}, held-out views {len(test)}", flush=True)\n\n    def render(v):\n        image, _, info = rasterization(\n            params["means"], params["quats"], params["scales"].exp(), params["opacities"].sigmoid(),\n            params["colors"].sigmoid(), v["view"], v["K"], v["w"], v["h"], packed=False,\n            camera_model=v["model"], radial_coeffs=v["radial"] if v["model"] == "fisheye" else None,\n            with_ut=True, with_eval3d=True)\n        return image[0], info\n\n    def gt(v):\n        return v["image"].float() / 255\n\n    def evaluate():\n        stats = {}\n        with torch.no_grad():\n            for v in test:\n                pred, _ = render(v)\n                err = (pred.clamp(0, 1) - gt(v)).square()\n                mse = err.mean().clamp_min(1e-10)\n                mse_m = ((err * v["eval_mask"]).sum() / (v["eval_mask"].sum() * 3).clamp_min(1)).clamp_min(1e-10)\n                stats.setdefault(v["cam"], []).append((float(-10 * torch.log10(mse)), float(-10 * torch.log10(mse_m))))\n        return {cam: {"full": float(np.mean([a for a, _ in x])), "masked": float(np.mean([b for _, b in x]))}\n                for cam, x in stats.items()}\n\n    def export(tag):\n        with torch.no_grad():\n            save_ply(OUT / f"gaussians{tag}.ply", params["means"].cpu().numpy(),\n                     torch.nn.functional.normalize(params["quats"], dim=-1).cpu().numpy(),\n                     params["scales"].cpu().numpy(), params["opacities"].cpu().numpy(),\n                     params["colors"].sigmoid().cpu().numpy())\n\n    def save_ckpt(next_step):\n        path, tmp = OUT / "ckpt.pt", OUT / "ckpt.pt.tmp"\n        torch.save({"step": next_step, "params": {k: v.detach().cpu() for k, v in params.items()},\n                    "opt": {k: o.state_dict() for k, o in optimizers.items()}}, tmp)\n        os.replace(tmp, path)\n        sync(path)\n\n    ssim_op = make_ssim() if USE_SSIM else None\n    t0 = time.perf_counter()\n    log = []\n    for step in range(start_step, STEPS):\n        v = random.choice(train)\n        progress = step / max(1, STEPS - 1)\n        optimizers["means"].param_groups[0]["lr"] = lrs["means"] * (0.01 ** progress)\n        pred, info = render(v)\n        target, m = gt(v), v["mask"]\n        loss = ((pred - target).abs() * m).sum() / (m.sum() * 3).clamp_min(1)\n        if USE_SSIM:\n            loss = 0.8 * loss + 0.2 * (1 - ssim_op(pred * m, target * m))\n        loss = loss + 0.01 * params["opacities"].sigmoid().mean() + 0.01 * params["scales"].exp().mean()\n        loss.backward()\n        radii = info["radii"]\n        visible = (radii[0] > 0).all(-1) if radii.ndim == 3 else radii[0] > 0\n        for opt in optimizers.values():\n            opt.step(visible)\n            opt.zero_grad(set_to_none=True)\n        strategy.step_post_backward(params, optimizers, state, step, info,\n                                    lr=optimizers["means"].param_groups[0]["lr"])\n        with torch.no_grad():\n            params["scales"].clamp_(np.log(0.005), np.log(MAX_SCALE))\n        if step == 0 or (step + 1) % 500 == 0:\n            rate = (step + 1 - start_step) / (time.perf_counter() - t0)\n            print(f"step {step + 1}/{STEPS} loss={loss.item():.4f} gaussians={len(params[\'means\'])} "\n                  f"visible={int(visible.sum())} {rate:.1f} it/s "\n                  f"peak_vram={torch.cuda.max_memory_allocated() / 2**30:.2f}GiB", flush=True)\n            log.append(dict(step=step + 1, loss=loss.item(), gaussians=len(params["means"])))\n        if CKPT_EVERY and (step + 1) % CKPT_EVERY == 0 and step + 1 < STEPS:\n            save_ckpt(step + 1)\n        if (step + 1) % 5000 == 0 and step + 1 < STEPS:\n            export("_partial")\n            sync(OUT / "gaussians_partial.ply")\n            if test:\n                print(f"[EVAL] step {step + 1} held-out PSNR {evaluate()}", flush=True)\n\n    metrics = {"psnr_per_camera": evaluate() if test else {}, "gaussians": len(params["means"]),\n               "frames": len(frame_ids), "steps": STEPS, "train_seconds": time.perf_counter() - t0, "log": log}\n    export("")\n    sync(OUT / "gaussians.ply")\n    for cam in ("image_00", "image_02", "image_03"):\n        v = next((x for x in test or train if x["cam"] == cam), None)\n        if v is not None:\n            with torch.no_grad():\n                pred, _ = render(v)\n            sheet = torch.cat([gt(v), pred.clamp(0, 1)], dim=1).cpu().numpy()\n            cv2.imwrite(str(OUT / f"compare_{cam}.png"), sheet[:, :, ::-1] * 255)\n            sync(OUT / f"compare_{cam}.png")\n    (OUT / "metrics.json").write_text(json.dumps({**metrics, "config": config}, indent=2))\n    sync(OUT / "metrics.json")\n    if Path(LOG_PATH).exists():\n        shutil.copyfile(LOG_PATH, OUT / "train.log")\n        sync(OUT / "train.log")\n    print(f"[DONE] {metrics[\'train_seconds\']:.0f}s | held-out PSNR {metrics[\'psnr_per_camera\']}", flush=True)\n    print("[OUT]", OUT, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/colab_cli_kitti360.py').write_text(trainer_source, encoding='utf-8')
print('Trainer written:', len(trainer_source), 'chars')


In [ ]:
#@title 5b. Kiểm tra phép chiếu LiDAR → camera (chạy trước khi train)
from pathlib import Path
checker_source = '"""KITTI-360 LiDAR -> camera projection sanity check (run before any long train).\n\nSelf-contained so it can be sent with `colab exec -f`. For one frame it projects\nthe Velodyne scan into all four cameras along three independent chains and saves\noverlays plus a numeric edge-alignment score:\n\n  A "official":  calib_cam_to_velo + calib_cam_to_pose + perspective.txt + Mei yaml\n                 (KITTI-360 devkit convention, treated as ground truth).\n  B "trainer":   rosa pkl `lidar2cam` + `camera_intrinsic`, exactly as\n                 colab_cli_kitti360.py builds its view matrices.\n  C "world":     scan of frame F is lifted to world with the trainer pose chain\n                 (cam0_to_world @ CAM_FRONT.lidar2cam) and re-projected into\n                 frame F+offset with the trainer view matrix. Only aligns if the\n                 multi-frame pose convention is right.\n\nEnv:\n  KITTI360_ARCHIVE   default /content/drive/MyDrive/datasets/kitti360_full\n  KITTI360_SEQ       default 2013_05_28_drive_0003_sync\n  KITTI360_FRAME     default: first frame present in both pkl and poses\n  KITTI360_OFFSET    default 5 (frame offset for chain C)\n  KITTI360_CHECK_OUT default /content/kitti360_projection_check\n"""\nimport json\nimport os\nimport pickle\nimport re\nimport tarfile\nimport urllib.request\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport cv2\nimport numpy as np\n\nARCHIVE = Path(os.environ.get("KITTI360_ARCHIVE", "/content/drive/MyDrive/datasets/kitti360_full"))\nHF = "https://huggingface.co/datasets/danny042/KITTI-360-fisheye/resolve/main"\nPOSES_URL = ("https://s3.eu-central-1.amazonaws.com/avg-projects/KITTI-360/"\n             "89a6bae3c8a6f789e12de4807fc1e8fdcf182cf4/data_poses.zip")\nSEQ = os.environ.get("KITTI360_SEQ", "2013_05_28_drive_0003_sync")\nOUT = Path(os.environ.get("KITTI360_CHECK_OUT", "/content/kitti360_projection_check"))\nOFFSET = int(os.environ.get("KITTI360_OFFSET", "5"))\n# (rosa pkl key, KITTI-360 camera folder, model) as used by the trainer.\nCAMERAS = [\n    ("CAM_FRONT", "image_00", "pinhole"),\n    ("CAM_FRONT_RIGHT", "image_01", "pinhole"),\n    ("CAM_BACK_RIGHT", "image_02", "fisheye"),\n    ("CAM_BACK_LEFT", "image_03", "fisheye"),\n]\n\n\n# ---------------------------------------------------------------- calibration\ndef read_records(path):\n    out = {}\n    for line in Path(path).read_text().splitlines():\n        key, sep, value = line.partition(":")\n        if sep and value.strip():\n            try:\n                out[key.strip()] = np.array([float(v) for v in value.split()])\n            except ValueError:\n                pass\n    return out\n\n\ndef to4(values):\n    m = np.eye(4)\n    m[:3] = np.asarray(values, dtype=float).reshape(3, 4)\n    return m\n\n\ndef read_mei_yaml(path):\n    text = Path(path).read_text()\n\n    def num(key):\n        match = re.search(rf"\\b{key}\\s*:\\s*([-+0-9.eE]+)", text)\n        if not match:\n            raise ValueError(f"{key} missing in {path}")\n        return float(match.group(1))\n\n    return dict(\n        xi=num("xi"), k1=num("k1"), k2=num("k2"), p1=num("p1"), p2=num("p2"),\n        K=np.array([[num("gamma1"), 0, num("u0")], [0, num("gamma2"), num("v0")], [0, 0, 1]]),\n        size=(int(num("image_width")), int(num("image_height"))),\n    )\n\n\ndef read_official_calibration(calib_dir):\n    """Return {cam: dict(T=cam_from_velo 4x4, model, K, ...)} in devkit convention."""\n    cam_to_pose = {k: to4(v) for k, v in read_records(calib_dir / "calib_cam_to_pose.txt").items()}\n    cam0_from_velo = np.linalg.inv(to4(np.loadtxt(calib_dir / "calib_cam_to_velo.txt").ravel()))\n    persp = read_records(calib_dir / "perspective.txt")\n    result = {}\n    for cam in ("image_00", "image_01", "image_02", "image_03"):\n        camk_from_cam0 = np.linalg.inv(cam_to_pose[cam]) @ cam_to_pose["image_00"]\n        unrect = camk_from_cam0 @ cam0_from_velo\n        if cam in ("image_00", "image_01"):\n            idx = cam[-2:]\n            r = np.eye(4)\n            r[:3, :3] = persp[f"R_rect_{idx}"].reshape(3, 3)\n            P = persp[f"P_rect_{idx}"].reshape(3, 4)\n            # P_rect_01 carries a baseline term in column 3; in devkit camera 1 is\n            # modeled with its own camToPose so only the 3x3 part is the intrinsic.\n            result[cam] = dict(model="pinhole", T=r @ unrect, K=P[:, :3],\n                               size=tuple(int(v) for v in persp[f"S_rect_{idx}"]))\n        else:\n            result[cam] = dict(model="mei", T=unrect, **read_mei_yaml(calib_dir / f"{cam}.yaml"))\n    rect0_from_velo = result["image_00"]["T"]\n    return result, rect0_from_velo\n\n\n# ----------------------------------------------------------------- projection\ndef project_pinhole(pts_cam, K):\n    z = pts_cam[:, 2]\n    valid = z > 0.5\n    uv = np.full((len(pts_cam), 2), np.nan)\n    uv[valid] = (pts_cam[valid, :2] / z[valid, None]) @ K[:2, :2].T + K[:2, 2]\n    return uv, valid\n\n\ndef project_mei(pts_cam, K, xi, k1, k2, p1=0.0, p2=0.0):\n    norm = np.linalg.norm(pts_cam, axis=1)\n    denom = pts_cam[:, 2] + xi * norm\n    valid = (norm > 0.5) & (denom > 1e-6)\n    uv = np.full((len(pts_cam), 2), np.nan)\n    x = pts_cam[valid, 0] / denom[valid]\n    y = pts_cam[valid, 1] / denom[valid]\n    r2 = x * x + y * y\n    radial = 1 + k1 * r2 + k2 * r2 * r2\n    xd = x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)\n    yd = y * radial + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y\n    uv[valid] = np.column_stack([K[0, 0] * xd + K[0, 2], K[1, 1] * yd + K[1, 2]])\n    return uv, valid\n\n\ndef project_equidistant(pts_cam, K, radial4):\n    """The model gsplat renders with: OpenCV fisheye, theta_d = theta(1+k1 t^2+...)."""\n    norm_xy = np.linalg.norm(pts_cam[:, :2], axis=1)\n    theta = np.arctan2(norm_xy, pts_cam[:, 2])\n    valid = (np.linalg.norm(pts_cam, axis=1) > 0.5) & (theta < np.pi / 2)\n    t2 = theta * theta\n    theta_d = theta * (1 + radial4[0] * t2 + radial4[1] * t2**2 + radial4[2] * t2**3 + radial4[3] * t2**4)\n    scale = np.where(norm_xy > 1e-9, theta_d / np.maximum(norm_xy, 1e-9), 1.0)\n    uv = np.column_stack([K[0, 0] * pts_cam[:, 0] * scale + K[0, 2], K[1, 1] * pts_cam[:, 1] * scale + K[1, 2]])\n    uv[~valid] = np.nan\n    return uv, valid\n\n\ndef mei_to_fisheye(xi, k1, k2):\n    """Copied from colab_cli_kitti360.py so chain B reproduces the trainer."""\n    theta = np.linspace(0.02, 1.45, 256)\n    u = np.sin(theta) / (np.cos(theta) + xi)\n    u2 = u * u\n    r = (1 + xi) * u * (1 + k1 * u2 + k2 * u2 * u2)\n    X = np.column_stack([theta**2, theta**4, theta**6, theta**8])\n    k, *_ = np.linalg.lstsq(X, r / theta - 1, rcond=None)\n    return k\n\n\ndef transform(T, pts):\n    return pts @ T[:3, :3].T + T[:3, 3]\n\n\n# -------------------------------------------------------------------- scoring\ndef edge_score(gray, uv, valid, depth):\n    """Mean edge strength under projected depth discontinuities vs. under all points.\n\n    Points whose depth jumps relative to their image neighbours should land on image\n    edges if alignment is good. Ratio ~1 means no correlation (misaligned); >1.3 is\n    typical of a correct calibration.\n    """\n    h, w = gray.shape\n    inside = valid & (uv[:, 0] >= 1) & (uv[:, 0] < w - 1) & (uv[:, 1] >= 1) & (uv[:, 1] < h - 1)\n    if inside.sum() < 200:\n        return dict(inside=int(inside.sum()), ratio=float("nan"))\n    edges = cv2.GaussianBlur(cv2.Canny(gray, 50, 150).astype(np.float32), (0, 0), 2.0)\n    px = uv[inside].astype(int)\n    d = depth[inside]\n    zbuf = np.full((h, w), np.inf, dtype=np.float32)\n    np.minimum.at(zbuf, (px[:, 1], px[:, 0]), d)\n    # Depth range inside an 11x11 window over the sparse z-buffer.\n    kernel = np.ones((11, 11), np.uint8)\n    near = cv2.erode(zbuf, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=np.inf)\n    far = cv2.dilate(np.where(np.isinf(zbuf), -np.inf, zbuf).astype(np.float32), kernel,\n                     borderType=cv2.BORDER_CONSTANT, borderValue=-np.inf)\n    jump = far - near\n    disc = np.nan_to_num(jump[px[:, 1], px[:, 0]]) > 0.1 * d.clip(1, None) + 0.5\n    all_mean = edges[px[:, 1], px[:, 0]].mean()\n    disc_mean = edges[px[disc, 1], px[disc, 0]].mean() if disc.any() else np.nan\n    return dict(inside=int(inside.sum()), discontinuities=int(disc.sum()),\n                ratio=float(disc_mean / max(all_mean, 1e-6)))\n\n\ndef overlay(bgr, uv, valid, depth, title):\n    img = bgr.copy()\n    h, w = img.shape[:2]\n    inside = valid & (uv[:, 0] >= 0) & (uv[:, 0] < w) & (uv[:, 1] >= 0) & (uv[:, 1] < h)\n    order = np.argsort(-depth[inside])\n    px = uv[inside][order].astype(int)\n    hue = (np.clip(np.log(depth[inside][order]) / np.log(60), 0, 1) * 120).astype(np.uint8)\n    colors = cv2.cvtColor(np.stack([hue, np.full_like(hue, 255), np.full_like(hue, 255)], -1)[None],\n                          cv2.COLOR_HSV2BGR)[0]\n    radius = max(1, round(w / 700))\n    for (u, v), c in zip(px, colors):\n        cv2.circle(img, (int(u), int(v)), radius, tuple(int(x) for x in c), -1)\n    cv2.putText(img, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 3)\n    cv2.putText(img, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 1)\n    return img\n\n\n# ----------------------------------------------------------------------- data\ndef fetch(url):\n    return urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=120)\n\n\ndef ensure_small_files():\n    """Download calibration, metadata and poses into ARCHIVE when missing (no Drive needed)."""\n    wanted = [f"calibration/{n}" for n in ("calib_cam_to_pose.txt", "calib_cam_to_velo.txt",\n                                           "perspective.txt", "image_02.yaml", "image_03.yaml")]\n    wanted.append("ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl")\n    for rel in wanted:\n        dst = ARCHIVE / rel\n        if not (dst.exists() and dst.stat().st_size > 0):\n            dst.parent.mkdir(parents=True, exist_ok=True)\n            with fetch(f"{HF}/{rel}") as src, dst.open("wb") as out:\n                out.write(src.read())\n            print("[DL]", rel, dst.stat().st_size, "bytes")\n    dst = ARCHIVE / "data_poses.zip"\n    if not (dst.exists() and dst.stat().st_size > 0):\n        with fetch(POSES_URL) as src, dst.open("wb") as out:\n            out.write(src.read())\n        print("[DL] data_poses.zip", dst.stat().st_size, "bytes")\n\n\ndef tar_members(tar_path, suffix, stems):\n    """Read members from a local tar, or stream them from Hugging Face when it is missing."""\n    out, names = {}, {}\n    tar_path = Path(tar_path)\n    if tar_path.exists():\n        source, mode = tar_path.open("rb"), "r:*"\n    else:\n        source, mode = fetch(f"{HF}/{tar_path.relative_to(ARCHIVE).as_posix()}"), "r|*"\n        print("[STREAM]", tar_path.name)\n    with source, tarfile.open(fileobj=source, mode=mode) as tar:\n        for m in tar:\n            if m.isfile() and m.name.endswith(suffix) and Path(m.name).stem in stems:\n                out.setdefault(Path(m.name).stem, tar.extractfile(m).read())\n                names.setdefault(Path(m.name).stem, []).append(m.name)\n                if len(out) == len(stems):\n                    break\n    return out, names\n\n\ndef read_cam0_to_world():\n    with ZipFile(ARCHIVE / "data_poses.zip") as z, z.open(f"{SEQ}/cam0_to_world.txt") as f:\n        poses = {}\n        for line in f:\n            v = np.array(line.split(), dtype=float)\n            if len(v) == 17:\n                poses[int(v[0])] = v[1:].reshape(4, 4)\n    return poses\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    ensure_small_files()\n    report = {"seq": SEQ}\n    official_cal, rect0_from_velo = read_official_calibration(ARCHIVE / "calibration")\n\n    with (ARCHIVE / "ImageSets-rosa-6class/kitti360_rosa_infos_val.pkl").open("rb") as f:\n        items = {x["frame_id"]: x for x in pickle.load(f)["data_list"] if SEQ in x["lidar_path"]}\n    poses = read_cam0_to_world()\n    common = sorted(set(items) & set(poses))\n    frame = int(os.environ.get("KITTI360_FRAME", common[0]))\n    later = next((f for f in common if f >= frame + OFFSET), None)\n    report.update(frame=frame, later_frame=later)\n    print(f"[DATA] frame {frame}, chain-C target frame {later}")\n\n    stems = {f"{frame:010d}"} | ({f"{later:010d}"} if later is not None else set())\n    images = {}\n    for _, cam, _ in CAMERAS:\n        data, names = tar_members(ARCHIVE / f"data_2d_raw/{SEQ}_{cam}.tar", ".png", stems)\n        images[cam] = {k: cv2.imdecode(np.frombuffer(v, np.uint8), cv2.IMREAD_COLOR) for k, v in data.items()}\n        report[f"{cam}_tar_members"] = names\n        print(f"[DATA] {cam}: {names}")\n    scans, names = tar_members(ARCHIVE / f"data_3d_raw/{SEQ}.tar", ".bin", {f"{frame:010d}"})\n    print(f"[DATA] lidar: {names}")\n    velo = np.frombuffer(scans[f"{frame:010d}"], np.float32).reshape(-1, 4)[:, :3].astype(float)\n    velo = velo[np.linalg.norm(velo, axis=1) > 2.5]\n\n    item = items[frame]\n    meta_front = np.array(item["images"]["CAM_FRONT"]["lidar2cam"], dtype=float)\n    report["pkl_img_paths"] = {key: item["images"][key].get("img_path") for key, _, _ in CAMERAS}\n    print("[META] pkl img_path per key:", json.dumps(report["pkl_img_paths"], indent=1))\n\n    rows, cam_rows = [], {}\n    for key, cam, model in CAMERAS:\n        bgr = images[cam][f"{frame:010d}"]\n        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n        info = item["images"][key]\n        cal = official_cal[cam]\n        res = {"image_size": [bgr.shape[1], bgr.shape[0]], "official_size": list(cal["size"])}\n\n        # A: official devkit chain.\n        pts = transform(cal["T"], velo)\n        uv_a, ok_a = (project_pinhole(pts, cal["K"]) if model == "pinhole"\n                      else project_mei(pts, cal["K"], cal["xi"], cal["k1"], cal["k2"], cal["p1"], cal["p2"]))\n        res["A_official"] = edge_score(gray, uv_a, ok_a, np.linalg.norm(pts, axis=1))\n\n        # B: trainer chain (pkl lidar2cam/intrinsics, equidistant fit for fisheye).\n        T_b = np.array(info["lidar2cam"], dtype=float)\n        K_b = np.array(info["camera_intrinsic"], dtype=float)\n        pts_b = transform(T_b, velo)\n        if model == "pinhole":\n            uv_b, ok_b = project_pinhole(pts_b, K_b)\n        else:\n            k1, k2, _, _, xi = info["distortion_coefficients"]\n            Kf = K_b.copy()\n            Kf[0, 0] /= 1 + xi\n            Kf[1, 1] /= 1 + xi\n            uv_b, ok_b = project_equidistant(pts_b, Kf, mei_to_fisheye(xi, k1, k2))\n            # B\': same pkl extrinsic but exact Mei, isolates the equidistant fit error.\n            uv_bm, ok_bm = project_mei(pts_b, K_b, xi, k1, k2)\n            both = ok_b & ok_bm\n            res["B_fit_vs_mei_px"] = float(np.nanmedian(np.linalg.norm(uv_b[both] - uv_bm[both], axis=1)))\n        res["B_trainer"] = edge_score(gray, uv_b, ok_b, np.linalg.norm(pts_b, axis=1))\n        res["T_pkl_minus_official_maxabs"] = float(np.abs(T_b - cal["T"]).max())\n        res["K_pkl"], res["K_official"] = K_b.round(3).tolist(), np.asarray(cal["K"]).round(3).tolist()\n        both = ok_a & ok_b\n        res["A_vs_B_median_px"] = float(np.nanmedian(np.linalg.norm(uv_a[both] - uv_b[both], axis=1))) if both.any() else None\n\n        panels = [overlay(bgr, uv_a, ok_a, np.linalg.norm(pts, axis=1), f"A official {cam}"),\n                  overlay(bgr, uv_b, ok_b, np.linalg.norm(pts_b, axis=1), f"B trainer {key}->{cam}")]\n\n        # C: multi-frame world consistency with the trainer pose chain.\n        if later is not None:\n            world_from_lidar = poses[frame] @ meta_front          # trainer: official @ front_lidar2cam\n            later_world_from_lidar = poses[later] @ np.array(items[later]["images"]["CAM_FRONT"]["lidar2cam"], dtype=float)\n            view = np.array(items[later]["images"][key]["lidar2cam"], dtype=float) @ np.linalg.inv(later_world_from_lidar)\n            pts_c = transform(view @ world_from_lidar, velo)\n            later_bgr = images[cam][f"{later:010d}"]\n            if model == "pinhole":\n                uv_c, ok_c = project_pinhole(pts_c, K_b)\n            else:\n                uv_c, ok_c = project_mei(pts_c, K_b, xi, k1, k2)\n            res["C_world_trainer"] = edge_score(cv2.cvtColor(later_bgr, cv2.COLOR_BGR2GRAY), uv_c, ok_c,\n                                                np.linalg.norm(pts_c, axis=1))\n            # C*: same with the documented convention world_from_velo = cam0_to_world @ rect0_from_velo.\n            view_ref = official_cal[cam]["T"] @ np.linalg.inv(poses[later] @ rect0_from_velo)\n            pts_r = transform(view_ref @ poses[frame] @ rect0_from_velo, velo)\n            uv_r, ok_r = (project_pinhole(pts_r, cal["K"]) if model == "pinhole"\n                          else project_mei(pts_r, cal["K"], cal["xi"], cal["k1"], cal["k2"], cal["p1"], cal["p2"]))\n            res["C_world_official"] = edge_score(cv2.cvtColor(later_bgr, cv2.COLOR_BGR2GRAY), uv_r, ok_r,\n                                                 np.linalg.norm(pts_r, axis=1))\n            panels += [overlay(later_bgr, uv_c, ok_c, np.linalg.norm(pts_c, axis=1), f"C trainer world {frame}->{later}"),\n                       overlay(later_bgr, uv_r, ok_r, np.linalg.norm(pts_r, axis=1), f"C official world {frame}->{later}")]\n\n        sheet = np.concatenate(panels, axis=0 if model == "pinhole" else 1)\n        if model == "fisheye":\n            sheet = cv2.resize(sheet, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)\n        cv2.imwrite(str(OUT / f"overlay_{cam}.png"), sheet)\n        cam_rows[cam] = res\n        print(f"[{cam}] " + json.dumps({k: v for k, v in res.items() if not k.startswith("K_")}))\n        rows.append(sheet)\n\n    report["cameras"] = cam_rows\n    (OUT / "projection_report.json").write_text(json.dumps(report, indent=2, default=str))\n    print("\\nEdge ratio ~1.0 = misaligned, >1.3 = aligned. Compare A (ground truth) with B/C.")\n    print("[OUT]", OUT)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/kitti360_projection_check.py').write_text(checker_source, encoding='utf-8')
%run /content/kitti360_projection_check.py

from IPython.display import display, Image
for cam in ('image_00', 'image_01', 'image_02', 'image_03'):
    p = Path('/content/kitti360_projection_check') / f'overlay_{cam}.png'
    if p.exists():
        print(p.name)
        display(Image(filename=str(p)))
print('Edge ratio ~1.0 = lệch; > 1.3 = khớp. B và C phải khớp A trước khi train.')


In [ ]:
#@title 6. Train (ghi kết quả + checkpoint lên Drive; KITTI360_RESUME=1 để chạy tiếp)
%run /content/colab_cli_kitti360.py


In [ ]:
#@title 7. Kiểm tra output
from pathlib import Path
from IPython.display import display, Image

out = Path('/content/kitti360_cli_out')
for p in sorted(out.glob('*')):
    print(p, p.stat().st_size, 'bytes')
for cam in ('image_00', 'image_02', 'image_03'):
    f = out / f'compare_{cam}.png'
    if f.exists():
        display(Image(filename=str(f)))


In [ ]:
#@title 8. Video fly-through (tùy chọn)
from pathlib import Path
fly_source = '"""Render a fly-through video from a trained Gaussian PLY (gsplat 3DGUT, front pinhole camera).\n\nCamera path = the CAM_FRONT poses of the training frames, smoothed with a cubic spline\n(position) and slerp (rotation). Intrinsics are the original full-resolution CAM_FRONT ones,\nso every pixel is inside the field of view the model was trained on.\n\nNeeds colab_cli_kitti360.py next to it, or FLY_TRAINER (reuses its pose/metadata loaders).\nEnv: FLY_PLY, FLY_OUT, FLY_FRAMES (100), FLY_SUB (10 interpolated frames per step), FLY_FPS (30),\n     KITTI360_ARCHIVE (Drive archive holding data_poses.zip and the metadata pickles).\n"""\nimport importlib.util\nimport json\nimport os\nimport pickle\nimport shutil\nimport subprocess\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom gsplat import rasterization\nfrom scipy.interpolate import CubicSpline\nfrom scipy.spatial.transform import Rotation, Slerp\n\nPLY = os.environ.get("FLY_PLY", "/content/run100_cap1m/gaussians.ply")\nOUT = Path(os.environ.get("FLY_OUT", "/content/fly/fly_best.mp4"))\nN_FRAMES = int(os.environ.get("FLY_FRAMES", "100"))\nSUB = int(os.environ.get("FLY_SUB", "10"))\nFPS = int(os.environ.get("FLY_FPS", "30"))\nos.environ.setdefault("KITTI360_ARCHIVE", "/content/drive/MyDrive/datasets/kitti360_full")\n\nos.environ["KITTI360_N_FRAMES"] = str(N_FRAMES)   # the trainer module reads it at import\nTRAINER = os.environ.get("FLY_TRAINER", str(Path(__file__).resolve().with_name("colab_cli_kitti360.py")))\nspec = importlib.util.spec_from_file_location("trainer", TRAINER)\ntrainer = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(trainer)\n\n\ndef load_ply(path):\n    with open(path, "rb") as f:\n        header = []\n        while True:\n            line = f.readline().decode("ascii").strip()\n            header.append(line)\n            if line == "end_header":\n                break\n        n = int(next(h for h in header if h.startswith("element vertex")).split()[2])\n        names = [h.split()[2] for h in header if h.startswith("property")]\n        data = np.frombuffer(f.read(), dtype="<f4").reshape(n, len(names))\n    col = {k: i for i, k in enumerate(names)}\n    pick = lambda keys: data[:, [col[k] for k in keys]].copy()\n    return dict(\n        means=pick(["x", "y", "z"]), quats=pick([f"rot_{i}" for i in range(4)]),\n        scales=pick([f"scale_{i}" for i in range(3)]), opacity=data[:, col["opacity"]].copy(),\n        colors=np.clip(pick([f"f_dc_{i}" for i in range(3)]) * 0.28209479 + 0.5, 0, 1))\n\n\ndef front_c2w_path():\n    """World-from-camera poses (same re-centred world as the trainer) and the front intrinsics."""\n    items = []\n    for split in ("val", "train"):\n        p = trainer.ensure_file(f"ImageSets-rosa-6class/kitti360_rosa_infos_{split}.pkl")\n        items += pickle.load(p.open("rb"))["data_list"]\n    official = trainer.read_official_poses()\n    cfg = Path(PLY).with_name("config.json")\n    used = json.loads(cfg.read_text()).get("frame_ids") if cfg.exists() else None\n    if used:                       # exactly the frames the model was trained on\n        by_id = {x["frame_id"]: x for x in items if trainer.SEQ in x["lidar_path"]}\n        missing = [f for f in used if f not in by_id or f not in official]\n        if missing:\n            raise SystemExit(f"config.json lists frames not found in the metadata/poses: {missing[:5]}")\n        items = [by_id[f] for f in used]\n        print(f"[FLY] frames from {cfg.name}: {used[0]}..{used[-1]} ({len(used)})", flush=True)\n    else:                          # older run without frame_ids: repeat the selection (see --max-gap)\n        items = trainer.select_items(items, official)\n        print("[FLY] no frame_ids in config.json; frames re-selected with the trainer\'s rule", flush=True)\n    front0 = np.array(items[0]["images"]["CAM_FRONT"]["lidar2cam"], dtype=np.float64)\n    origin = official[items[0]["frame_id"]][:3, 3].copy()\n    c2w = []\n    for it in items:\n        wfl = official[it["frame_id"]] @ front0\n        wfl[:3, 3] -= origin\n        view = np.array(it["images"]["CAM_FRONT"]["lidar2cam"], dtype=np.float64) @ np.linalg.inv(wfl)\n        c2w.append(np.linalg.inv(view))\n    K = np.array(items[0]["images"]["CAM_FRONT"]["camera_intrinsic"], dtype=np.float64)\n    return np.array(c2w), K, [it["frame_id"] for it in items]\n\n\ndef smooth(c2w, sub):\n    t = np.arange(len(c2w))\n    tq = np.linspace(0, len(c2w) - 1, (len(c2w) - 1) * sub + 1)\n    pos = CubicSpline(t, c2w[:, :3, 3])(tq)\n    rot = Slerp(t, Rotation.from_matrix(c2w[:, :3, :3]))(tq).as_matrix()\n    out = np.tile(np.eye(4), (len(tq), 1, 1))\n    out[:, :3, :3], out[:, :3, 3] = rot, pos\n    return out\n\n\ndef main():\n    OUT.parent.mkdir(parents=True, exist_ok=True)\n    g = load_ply(PLY)\n    print(f"[FLY] {len(g[\'means\'])} Gaussians from {PLY}", flush=True)\n    c2w, K, frame_ids = front_c2w_path()\n    path = smooth(c2w, SUB)\n    W, H = 1408, 376\n    print(f"[FLY] {len(frame_ids)} keyframes {frame_ids[0]}..{frame_ids[-1]} -> {len(path)} video frames "\n          f"at {W}x{H}, {len(path) / FPS:.0f}s", flush=True)\n\n    dev = "cuda"\n    means = torch.tensor(g["means"], device=dev)\n    quats = torch.nn.functional.normalize(torch.tensor(g["quats"], device=dev), dim=-1)\n    scales = torch.tensor(g["scales"], device=dev).exp()\n    opacity = torch.tensor(g["opacity"], device=dev).sigmoid()\n    colors = torch.tensor(g["colors"], device=dev)\n    Kt = torch.tensor(K, dtype=torch.float32, device=dev)[None]\n\n    raw = OUT.with_suffix(".raw.mp4")\n    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))\n    t0 = time.perf_counter()\n    with torch.no_grad():\n        for i, pose in enumerate(path):\n            view = torch.tensor(np.linalg.inv(pose), dtype=torch.float32, device=dev)[None]\n            img, _, _ = rasterization(means, quats, scales, opacity, colors, view, Kt, W, H, packed=False,\n                                      camera_model="pinhole", with_ut=True, with_eval3d=True)\n            frame = (img[0].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)\n            writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))\n            if i % 100 == 0:\n                print(f"[FLY] frame {i}/{len(path)} {(i + 1) / (time.perf_counter() - t0):.1f} fps", flush=True)\n    writer.release()\n\n    if shutil.which("ffmpeg"):   # re-encode to H.264 so the file plays in browsers and phones\n        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-c:v", "libx264", "-crf", "20",\n                        "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(OUT)], check=True)\n        raw.unlink()\n    else:\n        raw.replace(OUT)\n    print(f"[DONE] {OUT} {OUT.stat().st_size / 1e6:.1f} MB in {time.perf_counter() - t0:.0f}s", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/render_flythrough.py').write_text(fly_source, encoding='utf-8')
import os
os.environ['FLY_PLY'] = '/content/kitti360_cli_out/gaussians.ply'
os.environ['FLY_OUT'] = '/content/kitti360_cli_out/fly.mp4'
%run /content/render_flythrough.py
